In [2]:
from pyspark.sql.functions import *
from pyspark.sql import SparkSession

In [3]:
#now we create the session
spark = (
    SparkSession.builder
    .appName('OlistDataEng')
    .master("local[*]")
    .getOrCreate()   
)

print("Spark Version: ",spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/dipak/Desktop/pyspark-project/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/14 21:20:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/14 21:20:24 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark Version:  4.2.0


In [4]:
from pathlib import Path


order_dataset_path = Path.cwd().parent/'data'/'raw'/'olist_orders_dataset.csv'

print(order_dataset_path)

df = spark.read.format('csv')\
.option('inferSchema',True)\
.option('header',True)\
.load(str(order_dataset_path))

df.limit(10).show()

/Users/dipak/Desktop/pyspark-project/data/raw/olist_orders_dataset.csv
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|     

In [5]:
print(df.printSchema())

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

None


In [6]:
print('The number of record is: ',df.count())

The number of record is:  99441


In [7]:
#counting null values present in each column
col('order_id').isNull().sum()

TypeError: 'Column' object is not callable

In [ ]:
df.select(sum(col('order_id').isNull().cast('int')).alias('null_count')).show()

+----------+
|null_count|
+----------+
|         0|
+----------+



In [ ]:
print(df.columns)

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [ ]:
#now if we need to do for evrey columns
df.select([sum(col(c).isNull().cast('int')).alias(c)
          for c in df.columns]).show()

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [ ]:
#finding duplicates in the order_id
df1 = df.groupBy('order_id').count()
df1.filter(df1['count']>1).show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [ ]:
#finding the number of orders by their status
df.groupBy('order_status').count()\
.orderBy(col('count').desc())\
.show()

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



26/10/13 13:58:57 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 669946 ms exceeds timeout 120000 ms
26/10/13 13:58:57 WARN SparkContext: Killing executors is not supported by current scheduler.
26/10/13 13:59:06 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:85)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:707

In [8]:
df.groupBy('order_status').agg(count("*").alias("total_count"),count_distinct("customer_id").alias("distinct_count")).show()

+------------+-----------+--------------+
|order_status|total_count|distinct_count|
+------------+-----------+--------------+
|     shipped|       1107|          1107|
|    canceled|        625|           625|
|    approved|          2|             2|
|    invoiced|        314|           314|
|   delivered|      96478|         96478|
| unavailable|        609|           609|
|  processing|        301|           301|
|     created|          5|             5|
+------------+-----------+--------------+

